In [ ]:
# Cell 1
!pip install numpy==1.26.4 -q
!pip install torch-geometric -q

In [ ]:
# Cell 2
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool
import numpy as np
import pandas as pd
import os
import glob
from scipy import stats
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

DRIVE = '/content/drive/MyDrive/antibody_benchmark'
RESULTS_DIR = f'{DRIVE}/paegat_v1_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

BOLTZ_DIR = f'{DRIVE}/paegat_dataset'
OF3_DIR = f'{DRIVE}/of3_paegat_dataset'
PROT_DIR = f'{DRIVE}/protenix_paegat_dataset'

def load_graphs(dataset_dir):
    files = glob.glob(f'{dataset_dir}/*.pt')
    graphs = []
    for f in files:
        try:
            graphs.append(torch.load(f, weights_only=False))
        except:
            pass
    print(f"Loaded {len(graphs)} from {dataset_dir.split('/')[-1]}")
    return graphs

print("Loading datasets...")
boltz_graphs = load_graphs(BOLTZ_DIR)
of3_graphs = load_graphs(OF3_DIR)
prot_graphs = load_graphs(PROT_DIR)

# After loading all datasets, normalize attributes
for g in boltz_graphs + of3_graphs + prot_graphs:
    if hasattr(g, 'seed_dir'):
        del g.seed_dir
    if hasattr(g, 'seed'):
        del g.seed
print(f"Total: Boltz-2={len(boltz_graphs)}, OF3={len(of3_graphs)}, Protenix={len(prot_graphs)}")

class PAEGAT(nn.Module):
    def __init__(self, in_channels=23, hidden_channels=64, heads=4, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(in_channels, hidden_channels)
        self.conv1 = GATv2Conv(hidden_channels, hidden_channels // heads, heads=heads, edge_dim=1, dropout=dropout)
        self.conv2 = GATv2Conv(hidden_channels, hidden_channels // heads, heads=heads, edge_dim=1, dropout=dropout)
        self.output = nn.Sequential(
            nn.Linear(hidden_channels, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr, batch):
        x = F.relu(self.input_proj(x))
        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index, edge_attr))
        x = global_mean_pool(x, batch)
        return self.output(x).squeeze(-1)

def compute_ece(confidence, accuracy, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        mask = (confidence >= bins[i]) & (confidence < bins[i+1])
        if mask.sum() > 0:
            ece += mask.mean() * abs(confidence[mask].mean() - accuracy[mask].mean())
    return ece

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        loss = criterion(pred, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    preds, labels, iptms, pdbs = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            preds.extend(pred.cpu().numpy())
            labels.extend(batch.y.cpu().numpy())
            iptms.extend(batch.iptm.cpu().numpy())
            pdbs.extend(batch.pdb if isinstance(batch.pdb, list) else [batch.pdb])
    return np.array(preds), np.array(labels), np.array(iptms), pdbs


def run_round(round_num, train_graphs, test_graphs, test_model_name):
    print(f"\n{'='*50}")
    print(f"ROUND {round_num}: Train on {len(train_graphs)} graphs, Test on {test_model_name} ({len(test_graphs)} graphs)")
    print(f"{'='*50}")

    train_loader = DataLoader(train_graphs, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_graphs, batch_size=16, shuffle=False)

    model = PAEGAT().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.MSELoss()

    best_loss = float('inf')
    best_weights = None

    for epoch in range(50):
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        scheduler.step(train_loss)

        if train_loss < best_loss:
            best_loss = train_loss
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/50 | Loss: {train_loss:.4f}")

    # Load best weights
    model.load_state_dict(best_weights)
    torch.save(best_weights, f'{RESULTS_DIR}/paegat_v1_round{round_num}.pt')

    # Evaluate
    preds, labels, iptms, pdbs = evaluate(model, test_loader)
    acceptable = (labels >= 0.23).astype(float)

    ece_paegat = compute_ece(preds, acceptable)
    ece_iptm = compute_ece(iptms, acceptable)
    r_paegat, _ = stats.spearmanr(preds, labels)
    r_iptm, _ = stats.spearmanr(iptms, labels)

    high_conf_mask = iptms >= 0.7
    fpr_iptm = ((iptms >= 0.7) & (labels < 0.23)).sum() / max((iptms >= 0.7).sum(), 1)
    high_paegat_mask = preds >= 0.5
    fpr_paegat = ((preds >= 0.5) & (labels < 0.23)).sum() / max((preds >= 0.5).sum(), 1)

    print(f"\nRound {round_num} Results (test on {test_model_name}):")
    print(f"ECE   — PAE-GAT: {ece_paegat:.4f} vs ipTM: {ece_iptm:.4f} | improvement: {((ece_iptm-ece_paegat)/ece_iptm*100):.1f}%")
    print(f"r     — PAE-GAT: {r_paegat:.3f} vs ipTM: {r_iptm:.3f}")
    print(f"FPR   — PAE-GAT: {fpr_paegat:.3f} vs ipTM: {fpr_iptm:.3f}")

    # Save results
    df_results = pd.DataFrame({
        'pdb': pdbs,
        'dockq_ha': labels,
        'paegat_pred': preds,
        'iptm': iptms,
        'test_model': test_model_name,
        'round': round_num
    })
    df_results.to_csv(f'{RESULTS_DIR}/paegat_v1_round{round_num}_results.tsv', sep='\t', index=False)

    return {
        'round': round_num,
        'test_model': test_model_name,
        'ece_paegat': ece_paegat,
        'ece_iptm': ece_iptm,
        'r_paegat': r_paegat,
        'r_iptm': r_iptm,
        'fpr_paegat': fpr_paegat,
        'fpr_iptm': fpr_iptm,
        'ece_improvement_pct': (ece_iptm - ece_paegat) / ece_iptm * 100
    }




Device: cuda
Loading datasets...
Loaded 1109 from paegat_dataset
Loaded 1129 from of3_paegat_dataset
Loaded 1162 from protenix_paegat_dataset
Total: Boltz-2=1109, OF3=1129, Protenix=1162


In [ ]:
# ── TEMPORAL HOLDOUT EXPERIMENT ───────────────────────────────────────────────
# Addresses leakage concern: train on pre-2025, test on 2025+ structures

PDBS_PRE2025 = set(['7y3o', '7yds', '7yv1', '7z1x', '7zau', '7zoz', '8a44', '8acf', '8aci', '8aok', '8aom', '8apy', '8b17', '8b18', '8b19', '8b1a', '8b1d', '8b1e', '8b1f', '8b1g', '8b7h', '8be2', '8bec', '8bg2', '8bg3', '8bg4', '8bg5', '8bk2', '8byu', '8c3k', '8c5h', '8cbf', '8d9y', '8d9z', '8da0', '8ee2', '8eln', '8elq', '8et0', '8evd', '8ezk', '8ezl', '8ezm', '8f5i', '8f6u', '8f6v', '8fdo', '8fxv', '8g0i', '8g4p', '8gjr', '8gy5', '8gye', '8h5t', '8h5u', '8h63', '8h64', '8hes', '8hr2', '8idi', '8ido', '8im0', '8im1', '8ipc', '8jel', '8jeo', '8jyr', '8k33', '8k3k', '8k3l', '8opr', '8oww', '8oxv', '8oxw', '8oxx', '8oxy', '8ozb', '8pe1', '8pe2', '8pih', '8pmz', '8pwv', '8pww', '8pyr', '8q3j', '8q6r', '8q7o', '8q94', '8q95', '8qf4', '8qf5', '8qgx', '8qh0', '8qrg', '8qz3', '8r4u', '8rby', '8rj7', '8rp8', '8rw9', '8rz0', '8rz1', '8rz2', '8s0l', '8s0n', '8s6m', '8s79', '8sdf', '8sfs', '8sfx', '8sfz', '8sgn', '8siq', '8sk5', '8slb', '8sos', '8sow', '8soz', '8szy', '8tbb', '8tft', '8trs', '8ts0', '8tuz', '8tv3', '8u08', '8u3s', '8udz', '8uky', '8up2', '8urf', '8uvy', '8uw2', '8uw7', '8uw9', '8v52', '8v5q', '8vrs', '8vui', '8vvl', '8w0w', '8w90', '8weo', '8x0t', '8x0y', '8xk6', '8ybl', '8ybm', '8ybn', '8ybo', '8ybp', '8yj8', '8ysh', '8yvj', '8yvo', '8yxi', '8z8v', '9atm', '9au1', '9dx6', '9en2', '9erw', '9etl', '9fvb', '9fww', '9fzc', '9fzd', '9g2a', '9g2g', '9g48', '9g5l', '9g7k', '9jbq'])

PDBS_2025 = set(['8r4n', '8rwb', '8v13', '8vtd', '8vte', '8y9t', '8ykt', '8yub', '8yyz', '8z2e', '8z2v', '8z39', '8z3a', '8zca', '8zu9', '9azr', '9bdi', '9bt5', '9cds', '9d41', '9dbo', '9dl0', '9dq3', '9dq4', '9e7n', '9e7o', '9eht', '9ena', '9eze', '9ezh', '9ezi', '9ezu', '9ezv', '9ezw', '9f91', '9fc2', '9fik', '9fys', '9fyt', '9g97', '9ggp', '9ghi', '9ghj', '9glz', '9gox', '9gv3', '9gvl', '9gvn', '9gvo', '9h1f', '9h39', '9h4r', '9h4s', '9h6v', '9h77', '9hi4', '9hi6', '9hj9', '9hk1', '9hll', '9hml', '9ho4', '9ho5', '9ho6', '9hub', '9huo', '9hur', '9hxo', '9hzj', '9hzk', '9i59', '9iqp', '9jcy', '9jd0', '9jd1', '9jeb', '9jno', '9jqv', '9k6j', '9kur', '9l9y', '9lad', '9lh2', '9lme', '9lrf', '9m5b', '9m6j', '9m6o', '9mer', '9mev', '9mic', '9mid', '9mif', '9mj3', '9ml8', '9mqv', '9mxi', '9n8i', '9n8n', '9ncd', '9njy', '9npi', '9oar', '9obn', '9pi9', '9pru', '9pzi', '9q8l', '9qrv', '9rit', '9riu', '9rm2', '9rn6', '9th6', '9u8g', '9uk5', '9uvi', '9uzb', '9uzc', '9uzd', '9uze', '9uzf', '9uzg', '9uzh', '9uzi', '9vdy', '9vdz', '9vjj', '9vp0', '9y0a', '9yio'])

# Split graphs by deposition date
def split_graphs_by_date(graphs, pdbs_train, pdbs_test):
    train, test = [], []
    for g in graphs:
        pdb = g.pdb if isinstance(g.pdb, str) else g.pdb[0]
        pdb = pdb[:4].lower()
        if pdb in pdbs_train:
            train.append(g)
        elif pdb in pdbs_test:
            test.append(g)
    return train, test

b2_pre, b2_2025 = split_graphs_by_date(boltz_graphs, PDBS_PRE2025, PDBS_2025)
of3_pre, of3_2025 = split_graphs_by_date(of3_graphs, PDBS_PRE2025, PDBS_2025)
prot_pre, prot_2025 = split_graphs_by_date(prot_graphs, PDBS_PRE2025, PDBS_2025)

print("Temporal split summary:")
print(f"Boltz-2:   {len(b2_pre)} pre-2025 train | {len(b2_2025)} 2025+ test")
print(f"OpenFold3: {len(of3_pre)} pre-2025 train | {len(of3_2025)} 2025+ test")
print(f"Protenix:  {len(prot_pre)} pre-2025 train | {len(prot_2025)} 2025+ test")

# Save results to separate directory
TEMPORAL_DIR = f'{DRIVE}/paegat_temporal_results'
os.makedirs(TEMPORAL_DIR, exist_ok=True)

# Override RESULTS_DIR for this experiment
_orig_results_dir = RESULTS_DIR
RESULTS_DIR = TEMPORAL_DIR

temporal_summary = []

# T1: Train OF3+Protenix pre-2025 → Test Boltz-2 2025+
temporal_summary.append(run_round(1, of3_pre + prot_pre, b2_2025, 'Boltz-2'))

# T2: Train Boltz-2+Protenix pre-2025 → Test OF3 2025+
temporal_summary.append(run_round(2, b2_pre + prot_pre, of3_2025, 'OpenFold3'))

# T3: Train Boltz-2+OF3 pre-2025 → Test Protenix 2025+
temporal_summary.append(run_round(3, b2_pre + of3_pre, prot_2025, 'Protenix'))

# Restore original results dir
RESULTS_DIR = _orig_results_dir

# Summary
df_temporal = pd.DataFrame(temporal_summary)
df_temporal.to_csv(f'{TEMPORAL_DIR}/paegat_temporal_summary.tsv', sep='\t', index=False)

print(f"\n{'='*50}")
print("TEMPORAL HOLDOUT SUMMARY")
print(f"{'='*50}")
for _, row in df_temporal.iterrows():
    print(f"T{int(row['round'])} ({row['test_model']} 2025+): "
          f"ECE {row['ece_paegat']:.3f} vs {row['ece_iptm']:.3f} "
          f"| {row['ece_improvement_pct']:.1f}% improvement")

print(f"\nMean ECE PAE-GAT: {df_temporal['ece_paegat'].mean():.3f} +/- {df_temporal['ece_paegat'].std():.3f}")
print(f"Mean ECE ipTM:    {df_temporal['ece_iptm'].mean():.3f} +/- {df_temporal['ece_iptm'].std():.3f}")
print(f"Mean improvement: {df_temporal['ece_improvement_pct'].mean():.1f}%")

# Disconnect
from google.colab import runtime
runtime.unassign()

Temporal split summary:
Boltz-2:   617 pre-2025 train | 492 2025+ test
OpenFold3: 625 pre-2025 train | 504 2025+ test
Protenix:  630 pre-2025 train | 532 2025+ test

ROUND 1: Train on 1255 graphs, Test on Boltz-2 (492 graphs)
Epoch 10/50 | Loss: 0.0399
Epoch 20/50 | Loss: 0.0325
Epoch 30/50 | Loss: 0.0299
Epoch 40/50 | Loss: 0.0274
Epoch 50/50 | Loss: 0.0269

Round 1 Results (test on Boltz-2):
ECE   — PAE-GAT: 0.1041 vs ipTM: 0.3398 | improvement: 69.4%
r     — PAE-GAT: 0.434 vs ipTM: 0.579
FPR   — PAE-GAT: 0.398 vs ipTM: 0.459

ROUND 2: Train on 1247 graphs, Test on OpenFold3 (504 graphs)
Epoch 10/50 | Loss: 0.0413
Epoch 20/50 | Loss: 0.0405
Epoch 30/50 | Loss: 0.0378
Epoch 40/50 | Loss: 0.0351
Epoch 50/50 | Loss: 0.0345

Round 2 Results (test on OpenFold3):
ECE   — PAE-GAT: 0.1384 vs ipTM: 0.3102 | improvement: 55.4%
r     — PAE-GAT: 0.469 vs ipTM: 0.505
FPR   — PAE-GAT: 0.000 vs ipTM: 0.222

ROUND 3: Train on 1242 graphs, Test on Protenix (532 graphs)
Epoch 10/50 | Loss: 0.0466
Epoc